<table>
    <tr>
      <td>Minería de datos y Paradigma BigData (<b>MIN</b>) - Facultad de Informática - UCM
      </td>
      <td>
      <img src="https://biblioteca.ucm.es/data/cont/media/www/pag-88746//escudo.jpg"  width=50/>
      </td>
     </tr>
</table>




### Práctica 1 - Web scrapping de Ultimate Fighting Championship
Pablo C. Cañizares

No olvidéis los nombres de los dos componentes del grupo

### Se nos encarga preparar un dataset de luchadores de la compañía para futuras decisiones estratégicas
Entre estas decisiones radican posibles `matchmaking` de combates, herramientas para ojeadores, potenciación de carreras o incluso `¡despidos!`

Empezamos por acceder a la página de estadísicas de la reputada compañias de artes marciales mixtas UFC `ufcstats.com`. Específicamente, vamos a analizar inicialmente al vigente campeón del peso pluma el hispano-georgiano `Ilia "El matador" Topuria`. Para ello, accedemos al enlace `http://ufcstats.com/fighter-details/54f64b5e283b0ce7`

In [37]:
# (Solo ejecutar si hay problemas con la version de numpy)
#import sys
#!{sys.executable} -m pip uninstall -y pyarrow numexpr bottleneck streamlit


In [38]:
import pandas as pd
from bs4 import BeautifulSoup
import requests


def loadPage(urlIn):
    page = requests.get(urlIn)
    soup = BeautifulSoup(page.text, 'html.parser')  # le pasamos el texto en HTML para que lo analice
    return soup

url = "http://ufcstats.com/fighter-details/54f64b5e283b0ce7"
soup = loadPage(url) 
soup

<!DOCTYPE html>

<!--[if lt IE 7]>      <html class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html class="no-js ie8 lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js"> <!--<![endif]-->
<head>
<meta charset="utf-8"/>
<meta content="IE=edge" http-equiv="X-UA-Compatible"/>
<title>
    Stats | UFC
  </title>
<meta content="" name="description"/>
<meta content="" name="viewport"/>
<link href="/blocks/main.css?ver=191000" rel="stylesheet"/>
<script src="/js/vendor/modernizr-2.6.2.min.js"></script>
<script>
    (function(i,s,o,g,r,a,m){i['GoogleAnalyticsObject']=r;i[r]=i[r]||function(){
    (i[r].q=i[r].q||[]).push(arguments)},i[r].l=1*new Date();a=s.createElement(o),
    m=s.getElementsByTagName(o)[0];a.async=1;a.src=g;m.parentNode.insertBefore(a,m)
    })(window,document,'script','//www.google-analytics.com/analytics.js','ga');

    ga('create', 'UA-2855164-1', 'auto');
    ga('send

In [39]:
print(soup.head.title.text)


    Stats | UFC
  


**Ejercicio 1.** (1 punto) Extrae el nombre del luchador, así como su record y alias

In [40]:
# Nombre
def getName(soup):
    return soup.find("span", class_="b-content__title-highlight").text.strip()

# Record
def getRecord(soup):
    record_text = soup.find("span", class_="b-content__title-record").text
    return record_text.replace("Record:", "").strip()

# Nickname
def getNickName(soup):
    nick = soup.find("p", class_="b-content__Nickname")
    return nick.text.strip() if nick else ""

nombre = getName(soup)
record = getRecord(soup)
nickname= getNickName(soup)

print(record)
print(nombre)
print(nickname)

17-0-0
Ilia Topuria
El Matador


In [41]:
#Hacemos un diccionario con las stats iniciales
def getInitialStats(soup):
    dict ={"NAME":"","RECORD":"","NICKNAME":""}
    dict["NAME"] = getName(soup)
    dict["RECORD"] = getRecord(soup)
    dict["NICKNAME"] = getNickName(soup)
    return dict

initialStatsDict = getInitialStats(soup)
print(initialStatsDict)

{'NAME': 'Ilia Topuria', 'RECORD': '17-0-0', 'NICKNAME': 'El Matador'}


**Ejercicio 2.** (2 puntos) Extrae las caracteristicas básicas del luchador: HEIGHT, WEIGHT, REACH, STANCE, DOB

In [42]:
def cleanBasicStat(cadena):
    basicStat = cadena.split(":")[-1].strip()
    return basicStat

def getBasicStatsList(soup):
    cleanStats = ["", "", "", "", ""]

    items = soup.find_all("li", class_="b-list__box-list-item")

    for item in items:
        text = item.text.strip()

        if text.startswith("Height:"):
            cleanStats[0] = cleanBasicStat(text)
        elif text.startswith("Weight:"):
            cleanStats[1] = cleanBasicStat(text)
        elif text.startswith("Reach:"):
            cleanStats[2] = cleanBasicStat(text)
        elif text.startswith("STANCE:") or text.startswith("Stance:"):
            cleanStats[3] = cleanBasicStat(text)
        elif text.startswith("DOB:"):
            cleanStats[4] = cleanBasicStat(text)

    return cleanStats
cleanStats = getBasicStatsList(soup)
print(cleanStats)

['5\' 7"', '155 lbs.', '69"', 'Orthodox', 'Jan 21, 1997']


In [43]:
#Utiliza las funciones anteriores para extraer todos los datos
def getBasicStats(soup):
    #Obtenemos la lista de estadísticas básicas
    cleanStats = getBasicStatsList(soup)
    
    #Haz un diccionario con los valores
    basicStatsDict = {"HEIGHT":"", "WEIGHT":"", "REACH":"","STANCE":"","DOB":""}

    #Asignamos los valores (despues veremos un método más pro)
    basicStatsDict["HEIGHT"] = cleanStats[0]
    basicStatsDict["WEIGHT"] = cleanStats[1]
    basicStatsDict["REACH"] = cleanStats[2]
    basicStatsDict["STANCE"] = cleanStats[3]
    basicStatsDict["DOB"] = cleanStats[4]
    return basicStatsDict
    
basicStatsDict = getBasicStats(soup)
print(basicStatsDict)

{'HEIGHT': '5\' 7"', 'WEIGHT': '155 lbs.', 'REACH': '69"', 'STANCE': 'Orthodox', 'DOB': 'Jan 21, 1997'}


**Ejercicio 3.** (2 puntos) Extrae las caracteristicas de la carrera del luchador: 
- **SLpM** - Golpes significativos conectados por minuto.
- **Str. Acc.** - Precisión en golpes significativos.
- **SApM** - Golpes significativos absorbidos por minuto.
- **Str. Def.** - Defensa ante golpes significativos (porcentaje de golpes del oponente que no conectaron).
- **TD Avg.** - Derribos promedio logrados por cada 15 minutos.
- **TD Acc.** - Precisión en derribos.
- **TD Def.** - Defensa ante derribos (porcentaje de intentos de derribo del oponente que no tuvieron éxito).
- **Sub. Avg.** - Intentos de sumisión promedio por cada 15 minutos.

In [44]:
#Obten solo los resultados, en el siguiente apartado los adaptamos con diccionarios
def getCareerStatsList(soup):
    cleanCareerStats = []
    
    # Seleccionamos todos los items de estadísticas de carrera
    items = soup.select(
        ".b-list__info-box-left li.b-list__box-list-item, "
        ".b-list__info-box-right li.b-list__box-list-item"
    )
    
    for item in items:
        # Extraemos solo el valor, quitando el label
        value = item.text.split(":")[-1].strip()
        cleanCareerStats.append(value)
    
    return cleanCareerStats

cleanCareerStats = getCareerStatsList(soup)
print(cleanCareerStats)

['4.81', '48%', '3.83', '64%', '', '1.96', '61%', '93%', '1.1']


In [45]:
def getCareerStats(soup):
    cleanCareerStats = getCareerStatsList(soup)
    #Hacemos otro diccionario
    dictCS = {"SLpM":"", "Str. Acc.":"", "SApM":"","Str. Def.":"","TD Avg.":"", "TD Acc.":"", "TD Def.":"", "Sub. Avg.":""}
    dictCS = dict(zip(dictCS.keys(), cleanCareerStats))
    return dictCS
    
careerStatsDict =  getCareerStats(soup)  
print(careerStatsDict)

{'SLpM': '4.81', 'Str. Acc.': '48%', 'SApM': '3.83', 'Str. Def.': '64%', 'TD Avg.': '', 'TD Acc.': '1.96', 'TD Def.': '61%', 'Sub. Avg.': '93%'}


**Ejercicio 4** (1 punto). El siguiente paso es unificar las funciones anteriores. Crea una función *extractFighterStats* que obtenga una lista de las estadísticas extraidas de la página del luchador proporcionada.

Para ello, vamos a tener que es unificar el diccionario. Existen tres metodos distintos, vamos a utilizar el más directo y más elegante

In [46]:
#Asi se mezclan los diccionarios
fighterDict = {**initialStatsDict, **basicStatsDict, **careerStatsDict}
print(fighterDict)

#Y así se obtienen las claves y valores. Puede que te sirva para despues:
print(fighterDict.keys())
print(fighterDict.values())

{'NAME': 'Ilia Topuria', 'RECORD': '17-0-0', 'NICKNAME': 'El Matador', 'HEIGHT': '5\' 7"', 'WEIGHT': '155 lbs.', 'REACH': '69"', 'STANCE': 'Orthodox', 'DOB': 'Jan 21, 1997', 'SLpM': '4.81', 'Str. Acc.': '48%', 'SApM': '3.83', 'Str. Def.': '64%', 'TD Avg.': '', 'TD Acc.': '1.96', 'TD Def.': '61%', 'Sub. Avg.': '93%'}
dict_keys(['NAME', 'RECORD', 'NICKNAME', 'HEIGHT', 'WEIGHT', 'REACH', 'STANCE', 'DOB', 'SLpM', 'Str. Acc.', 'SApM', 'Str. Def.', 'TD Avg.', 'TD Acc.', 'TD Def.', 'Sub. Avg.'])
dict_values(['Ilia Topuria', '17-0-0', 'El Matador', '5\' 7"', '155 lbs.', '69"', 'Orthodox', 'Jan 21, 1997', '4.81', '48%', '3.83', '64%', '', '1.96', '61%', '93%'])


In [54]:
def extractFighterStats(urlIn):
    import requests
    from bs4 import BeautifulSoup

    # Obtenemos la página con soup
    soup = BeautifulSoup(requests.get(urlIn).text, 'html.parser')
    
    # Adquirimos las listas de estadísticas
    initialStatsList = initialStatsList = list(getInitialStats(soup).values())
    basicStatsList = getBasicStatsList(soup)
    careerStatsList = getCareerStatsList(soup)
    
    # Unimos las listas en una sola
    listFighter = initialStatsList + basicStatsList + careerStatsList
    
    # Devolvemos la lista
    return listFighter

listFighter = extractFighterStats(url)
print(listFighter)

['Ilia Topuria', '17-0-0', 'El Matador', '5\' 7"', '155 lbs.', '69"', 'Orthodox', 'Jan 21, 1997', '4.81', '48%', '3.83', '64%', '', '1.96', '61%', '93%', '1.1']


**Ejercicio 5** (1 puntos). Ahora vamos a probar a extraer los datos de varios luchadores. Explora la página `http://ufcstats.com/statistics/fighters`, la cual lista distintos peleadores, extrae todos los enlaces y carga las stats de los nombres. 

In [50]:
urlFighters = "http://ufcstats.com/statistics/fighters"
page = requests.get(urlFighters)
soup = BeautifulSoup(page.text, 'html.parser')  

In [51]:
def extractFighterLinksList(soup):
    retList = []
    # Buscamos todos los enlaces de los luchadores
    for a in soup.select('a.b-link[href*="fighter-details"]'):
        retList.append(a['href'])
    return retList
                
linkList = extractFighterLinksList(soup)
print(linkList)

['http://ufcstats.com/fighter-details/93fe7332d16c6ad9', 'http://ufcstats.com/fighter-details/93fe7332d16c6ad9', 'http://ufcstats.com/fighter-details/93fe7332d16c6ad9', 'http://ufcstats.com/fighter-details/15df64c02b6b0fde', 'http://ufcstats.com/fighter-details/15df64c02b6b0fde', 'http://ufcstats.com/fighter-details/15df64c02b6b0fde', 'http://ufcstats.com/fighter-details/59a9d6dac61c2540', 'http://ufcstats.com/fighter-details/59a9d6dac61c2540', 'http://ufcstats.com/fighter-details/59a9d6dac61c2540', 'http://ufcstats.com/fighter-details/4961467134abd8be', 'http://ufcstats.com/fighter-details/4961467134abd8be', 'http://ufcstats.com/fighter-details/4961467134abd8be', 'http://ufcstats.com/fighter-details/b361180739bed4b0', 'http://ufcstats.com/fighter-details/b361180739bed4b0', 'http://ufcstats.com/fighter-details/b361180739bed4b0', 'http://ufcstats.com/fighter-details/3329d692aea4dc28', 'http://ufcstats.com/fighter-details/3329d692aea4dc28', 'http://ufcstats.com/fighter-details/3329d692ae

**Ejercicio 6 (1 punto)**. Es momento de hacer nuestro mini dataset de luchadores. Para ello, itera la lista de enlaces, y extrae las stats de todos ellos. Cada vez que extraigas stats de un peleador, pon tu programa a dormir por 1 segundo ( utiliza `time.sleep(1)`)

In [55]:
import time

def extractFighterListOfLists(linkList):
    lists=[]
    i=0
    for link in linkList:
        print("link: ", link)
        stats = extractFighterStats(link)  
        lists.append(stats)                 
        print("Extracted {0} stat: ".format(i),stats)
        time.sleep(1)                      
        i = i+1
    return lists

listOfLists = extractFighterListOfLists(linkList)

link:  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
Extracted 0 stat:  ['Tom Aaron', '5-3-0', '', '--', '155 lbs.', '--', '', 'Jul 13, 1978', '0.00', '0%', '0.00', '0%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
Extracted 1 stat:  ['Tom Aaron', '5-3-0', '', '--', '155 lbs.', '--', '', 'Jul 13, 1978', '0.00', '0%', '0.00', '0%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
Extracted 2 stat:  ['Tom Aaron', '5-3-0', '', '--', '155 lbs.', '--', '', 'Jul 13, 1978', '0.00', '0%', '0.00', '0%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/15df64c02b6b0fde
Extracted 3 stat:  ['Danny Abbadi', '4-6-0', 'The Assassin', '5\' 11"', '155 lbs.', '--', 'Orthodox', 'Jul 03, 1983', '3.29', '38%', '4.41', '57%', '', '0.00', '0%', '77%', '0.0']
link:  http://ufcstats.com/fighter-details/15df64c02b6b0fde
Extracted 4 stat:  ['Danny Abbadi', '4-6-0', 'The Assassin', '5\' 11"', '

link:  http://ufcstats.com/fighter-details/aa6e591c2a2cdecd
Extracted 35 stat:  ['Ricardo Abreu', '5-1-0', 'Demente', '5\' 11"', '185 lbs.', '--', 'Orthodox', 'Apr 27, 1984', '3.79', '31%', '3.98', '68%', '', '2.13', '42%', '100%', '0.7']
link:  http://ufcstats.com/fighter-details/7279654c7674cd24
Extracted 36 stat:  ['Klidson Abreu', '15-4-0 (1 NC)', 'White Bear', '6\' 0"', '205 lbs.', '74"', 'Orthodox', 'Dec 24, 1992', '2.05', '40%', '2.90', '55%', '', '0.64', '20%', '80%', '0.0']
link:  http://ufcstats.com/fighter-details/7279654c7674cd24
Extracted 37 stat:  ['Klidson Abreu', '15-4-0 (1 NC)', 'White Bear', '6\' 0"', '205 lbs.', '74"', 'Orthodox', 'Dec 24, 1992', '2.05', '40%', '2.90', '55%', '', '0.64', '20%', '80%', '0.0']
link:  http://ufcstats.com/fighter-details/7279654c7674cd24
Extracted 38 stat:  ['Klidson Abreu', '15-4-0 (1 NC)', 'White Bear', '6\' 0"', '205 lbs.', '74"', 'Orthodox', 'Dec 24, 1992', '2.05', '40%', '2.90', '55%', '', '0.64', '20%', '80%', '0.0']
link:  http://

Extracted 70 stat:  ['Nick Agallar', '24-6-0', '', '5\' 8"', '155 lbs.', '--', 'Orthodox', 'Jan 13, 1979', '0.69', '11%', '4.56', '42%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/ebc5af72ad5a28cb
Extracted 71 stat:  ['Nick Agallar', '24-6-0', '', '5\' 8"', '155 lbs.', '--', 'Orthodox', 'Jan 13, 1979', '0.69', '11%', '4.56', '42%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/a08ddd04eaffd81d
Extracted 72 stat:  ['Mariya Agapova', '10-5-0', 'Money Mashka', '5\' 6"', '125 lbs.', '68"', 'Southpaw', 'Apr 07, 1997', '4.43', '54%', '3.62', '52%', '', '0.55', '66%', '45%', '0.8']
link:  http://ufcstats.com/fighter-details/a08ddd04eaffd81d
Extracted 73 stat:  ['Mariya Agapova', '10-5-0', 'Money Mashka', '5\' 6"', '125 lbs.', '68"', 'Southpaw', 'Apr 07, 1997', '4.43', '54%', '3.62', '52%', '', '0.55', '66%', '45%', '0.8']
link:  http://ufcstats.com/fighter-details/a08ddd04eaffd81d
Extracted 74 stat:  ['Mariya Agapova', '10-5-0', 'Mone

**Ejercicio 7 (1 punto)**. Ahora te toca crear un dataset, y guardarlo con el nombre `fighters.csv`

In [62]:
#Primero creamos el DF
listOfLists = extractFighterListOfLists(linkList)

df_fighters =  pd.DataFrame(listOfLists)
print(df_fighters)

link:  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
Extracted 0 stat:  ['Tom Aaron', '5-3-0', '', '--', '155 lbs.', '--', '', 'Jul 13, 1978', '0.00', '0%', '0.00', '0%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
Extracted 1 stat:  ['Tom Aaron', '5-3-0', '', '--', '155 lbs.', '--', '', 'Jul 13, 1978', '0.00', '0%', '0.00', '0%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/93fe7332d16c6ad9
Extracted 2 stat:  ['Tom Aaron', '5-3-0', '', '--', '155 lbs.', '--', '', 'Jul 13, 1978', '0.00', '0%', '0.00', '0%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/15df64c02b6b0fde
Extracted 3 stat:  ['Danny Abbadi', '4-6-0', 'The Assassin', '5\' 11"', '155 lbs.', '--', 'Orthodox', 'Jul 03, 1983', '3.29', '38%', '4.41', '57%', '', '0.00', '0%', '77%', '0.0']
link:  http://ufcstats.com/fighter-details/15df64c02b6b0fde
Extracted 4 stat:  ['Danny Abbadi', '4-6-0', 'The Assassin', '5\' 11"', '

link:  http://ufcstats.com/fighter-details/aa6e591c2a2cdecd
Extracted 35 stat:  ['Ricardo Abreu', '5-1-0', 'Demente', '5\' 11"', '185 lbs.', '--', 'Orthodox', 'Apr 27, 1984', '3.79', '31%', '3.98', '68%', '', '2.13', '42%', '100%', '0.7']
link:  http://ufcstats.com/fighter-details/7279654c7674cd24
Extracted 36 stat:  ['Klidson Abreu', '15-4-0 (1 NC)', 'White Bear', '6\' 0"', '205 lbs.', '74"', 'Orthodox', 'Dec 24, 1992', '2.05', '40%', '2.90', '55%', '', '0.64', '20%', '80%', '0.0']
link:  http://ufcstats.com/fighter-details/7279654c7674cd24
Extracted 37 stat:  ['Klidson Abreu', '15-4-0 (1 NC)', 'White Bear', '6\' 0"', '205 lbs.', '74"', 'Orthodox', 'Dec 24, 1992', '2.05', '40%', '2.90', '55%', '', '0.64', '20%', '80%', '0.0']
link:  http://ufcstats.com/fighter-details/7279654c7674cd24
Extracted 38 stat:  ['Klidson Abreu', '15-4-0 (1 NC)', 'White Bear', '6\' 0"', '205 lbs.', '74"', 'Orthodox', 'Dec 24, 1992', '2.05', '40%', '2.90', '55%', '', '0.64', '20%', '80%', '0.0']
link:  http://

Extracted 70 stat:  ['Nick Agallar', '24-6-0', '', '5\' 8"', '155 lbs.', '--', 'Orthodox', 'Jan 13, 1979', '0.69', '11%', '4.56', '42%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/ebc5af72ad5a28cb
Extracted 71 stat:  ['Nick Agallar', '24-6-0', '', '5\' 8"', '155 lbs.', '--', 'Orthodox', 'Jan 13, 1979', '0.69', '11%', '4.56', '42%', '', '0.00', '0%', '0%', '0.0']
link:  http://ufcstats.com/fighter-details/a08ddd04eaffd81d
Extracted 72 stat:  ['Mariya Agapova', '10-5-0', 'Money Mashka', '5\' 6"', '125 lbs.', '68"', 'Southpaw', 'Apr 07, 1997', '4.43', '54%', '3.62', '52%', '', '0.55', '66%', '45%', '0.8']
link:  http://ufcstats.com/fighter-details/a08ddd04eaffd81d
Extracted 73 stat:  ['Mariya Agapova', '10-5-0', 'Money Mashka', '5\' 6"', '125 lbs.', '68"', 'Southpaw', 'Apr 07, 1997', '4.43', '54%', '3.62', '52%', '', '0.55', '66%', '45%', '0.8']
link:  http://ufcstats.com/fighter-details/a08ddd04eaffd81d
Extracted 74 stat:  ['Mariya Agapova', '10-5-0', 'Mone

In [63]:
#Ahora lo guardamos
df_fighters.to_csv('fighters.csv', sep=',', index=False)
print("Archivo CSV 'df_fighters.csv' creado.")

Archivo CSV 'df_fighters.csv' creado.


**Ejercicio 8** (1 punto). Ahora, vamos a intentar crear un dataset con los luchadores más importantes. Para ello, accede a la página `http://ufcstats.com/statistics/fighters?char=a&page=all` y selecciona únicamente los campeones. Cabe destacar que los campeones son aquellos que tienen el icono de un cinturón <img src="http://1e49bc5171d173577ecd-1323f4090557a33db01577564f60846c.r80.cf1.rackcdn.com/belt.png" class="b-list__icon"> en la última celda

In [66]:
# Lista de links de los campeones
champion_links = []

for row in rows:
    belt_cell = row.find_all("td")[-1]  # última celda
    if belt_cell.find("i", class_="b-statistics__icon-champion"):  # solo si hay cinturón
        link = row.find("a")["href"]  # link del fighter
        champion_links.append(link)

# Extraemos las estadísticas de cada campeón
listOfLists = extractFighterListOfLists(champion_links)

# Mostramos la lista final
print(listOfLists)

NameError: name 'rows' is not defined

**Ejercicio extra 1** Si ya has llegado hasta aqui, tienes que intentarlo, en el ejercicio anterior únicamente has obtenido los campeones que empiezan por la letra a. Obtiene todos los campeones de la 'a' a la 'z', modificando el caracter XX por una letra ``http://ufcstats.com/statistics/fighters?char=XX&page=all``

In [ ]:
champions = []
# ...
    
print(champions)

In [ ]:
#Creamos el DF
df_champs = pd.DataFrame(champions, columns=['NAME', 'RECORD', 'NICKNAME', 'HEIGHT', 'WEIGHT', 'REACH', 'STANCE', 'DOB', 'SLpM', 'Str. Acc.', 'SApM', 'Str. Def.', 'TD Avg.', 'TD Acc.', 'TD Def.', 'Sub. Avg.'])
print(df_champs)
#Y lo guardamos
df_champs.to_csv('fighters_champs.csv', sep=',', index=False)
print("Archivo CSV 'df_champs.csv' creado.")

**Ejercicio extra 2** Extrae todos los luchadores, no solo campeones

In [ ]:
fighters = []
# ...
    
print(fighters)


#Creamos el DF
df_fighters = pd.DataFrame(fighters, columns=['NAME', 'RECORD', 'NICKNAME', 'HEIGHT', 'WEIGHT', 'REACH', 'STANCE', 'DOB', 'SLpM', 'Str. Acc.', 'SApM', 'Str. Def.', 'TD Avg.', 'TD Acc.', 'TD Def.', 'Sub. Avg.'])
print(df_fighters)
#Y lo guardamos
df_fighters.to_csv('all_fighters.csv', sep=',', index=False)
print("Archivo CSV 'all_fighters.csv' creado.")


**Ejercicio extra 3** Extrae todas las peleas del evento UFC 308 con sus detalles y haz un dataframe `ufc308.csv`

In [ ]:
# ...